# Mimi-VC Demo: Voice Conversion with FAISS+Smooth

This notebook lets you **convert any voice to sound like a target speaker** using the best method from our paper:
**Mimi-VC (FAISS+Smooth)** — training-free, real-time, retrieval-based voice conversion.

### How it works
1. **Record or upload target speaker audio** (the voice you want to clone)
2. **Record or upload source audio** (the speech you want to convert)
3. The system preserves **what** is said (semantic content) and changes **how** it sounds (speaker identity)

### Steps
1. Select GPU runtime: **Runtime > Change runtime type > T4 GPU**
2. Run all cells in order
3. Record your audio directly in the browser when prompted

> Based on: *Mimi-VC: Voice Conversion via Disentangled Neural Audio Codec Token Manipulation*
> — Q.V. Nguyen, H. Nguyen, L.D. Vu

In [ ]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================
!pip install -q transformers torch torchaudio faiss-cpu librosa soundfile matplotlib
print('Dependencies installed.')

In [ ]:
# ============================================================
# 2. IMPORTS & LOAD MIMI CODEC
# ============================================================
import torch
import torch.nn.functional as F
import torchaudio
import numpy as np
import faiss
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import os
import time
from collections import defaultdict
from pathlib import Path
from transformers import MimiModel, AutoFeatureExtractor
from IPython.display import Audio as PlayAudio, display, HTML
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Load Mimi codec
print('Loading Mimi codec (this takes ~30s on first run)...')
mimi = MimiModel.from_pretrained('kyutai/mimi').to(device).eval()
feat_ext = AutoFeatureExtractor.from_pretrained('kyutai/mimi')

SR = feat_ext.sampling_rate           # 24000
N_CB = mimi.config.num_quantizers     # 8
CB_SIZE = mimi.config.codebook_size   # 2048
FRAME_RATE = mimi.config.frame_rate   # 12.5

print(f'Mimi loaded: SR={SR}Hz, {N_CB} codebooks x {CB_SIZE}, {FRAME_RATE}Hz frame rate')
print(f'  CB0 = semantic (content)  |  CB1-7 = acoustic (speaker identity)')

In [ ]:
# ============================================================
# 3. CORE FUNCTIONS
# ============================================================

def load_audio(path, target_sr=SR):
    """Load audio file, convert to mono, resample to target SR."""
    wav, orig_sr = librosa.load(path, sr=None, mono=True)
    if orig_sr != target_sr:
        wav = librosa.resample(wav, orig_sr=orig_sr, target_sr=target_sr)
    return wav.astype(np.float32)


def encode(wav_np):
    """Encode numpy audio to Mimi tokens [1, N_CB, T]."""
    inputs = feat_ext(raw_audio=wav_np, sampling_rate=SR, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = mimi.encode(inputs['input_values'], inputs.get('padding_mask'))
    return out.audio_codes.cpu()


def decode(codes):
    """Decode Mimi tokens to numpy audio."""
    with torch.no_grad():
        out = mimi.decode(codes.to(device))
    return out.audio_values.cpu().squeeze().numpy()


def get_codebook_embeddings(cb_idx=0):
    """Extract codebook embedding matrix from Mimi's split quantizer."""
    q = mimi.quantizer
    if hasattr(q, 'semantic_residual_vector_quantizer'):
        n_sem = q.num_semantic_quantizers
        if cb_idx < n_sem:
            layer = q.semantic_residual_vector_quantizer.layers[cb_idx]
        else:
            layer = q.acoustic_residual_vector_quantizer.layers[cb_idx - n_sem]
    elif hasattr(q, 'layers'):
        layer = q.layers[cb_idx]
    else:
        raise RuntimeError(f'Unknown quantizer: {type(q).__name__}')

    for attr in ['codebook.embed', 'codebook.embed_sum', 'codebook.embedding.weight']:
        obj = layer
        try:
            for part in attr.split('.'):
                obj = getattr(obj, part)
            w = obj.detach().cpu().float()
            if w.dim() == 3:
                w = w.squeeze(0)
            if w.shape[0] >= CB_SIZE:
                return w
        except AttributeError:
            continue
    raise RuntimeError('Cannot find codebook embeddings')


# Pre-load CB0 embeddings
cb0_emb = get_codebook_embeddings(0).numpy()
print(f'CB0 embeddings loaded: {cb0_emb.shape}')
print('Core functions ready.')

In [ ]:
# ============================================================
# 4. MIMI-VC (FAISS+SMOOTH) ENGINE
# ============================================================

class MimiVCEngine:
    """Voice conversion engine using FAISS retrieval + temporal smoothing."""

    def __init__(self):
        self.is_ready = False
        self.target_name = None
        self.n_frames = 0

    def build_index(self, target_wavs, target_name='target'):
        """
        Build FAISS index from target speaker audio.

        Args:
            target_wavs: list of numpy audio arrays (24kHz, mono)
            target_name: label for the target speaker
        """
        self.target_name = target_name
        print(f'Building voice index for "{target_name}"...')

        # Encode all target utterances
        all_sem = []
        all_full = []

        for i, wav in enumerate(target_wavs):
            codes = encode(wav)  # [1, 8, T]
            T = codes.shape[2]
            for t in range(T):
                all_sem.append(codes[0, 0, t].item())
                all_full.append(codes[0, :, t].numpy())
            print(f'  Encoded chunk {i+1}/{len(target_wavs)}: {T} frames ({T/FRAME_RATE:.1f}s)')

        self.sem_arr = np.array(all_sem)
        self.full_arr = np.stack(all_full)
        self.n_frames = len(all_sem)

        # Build FAISS index on CB0 embeddings
        tgt_emb = cb0_emb[self.sem_arr].astype(np.float32)
        faiss.normalize_L2(tgt_emb)
        self.faiss_index = faiss.IndexFlatIP(cb0_emb.shape[1])
        self.faiss_index.add(tgt_emb)

        # Also build lookup table for exact matches
        self.lookup = defaultdict(list)
        for i, s in enumerate(all_sem):
            self.lookup[s].append(i)

        self.is_ready = True
        total_dur = self.n_frames / FRAME_RATE
        print(f'Index built: {self.n_frames} frames ({total_dur:.1f}s), '
              f'{len(self.lookup)} unique semantic tokens')

    def convert(self, source_wav, k=8, alpha=0.5):
        """
        Convert source audio to target speaker voice.

        Args:
            source_wav: numpy audio (24kHz, mono)
            k: number of FAISS neighbors
            alpha: smoothing weight (0=temporal, 1=semantic)
        Returns:
            converted_wav: numpy audio
        """
        if not self.is_ready:
            raise RuntimeError('Build index first with build_index()')

        t0 = time.time()

        # Encode source
        src_codes = encode(source_wav)
        T = src_codes.shape[2]

        # Embed source semantic tokens
        src_sem = src_codes[0, 0].numpy()
        src_emb = cb0_emb[src_sem].astype(np.float32)
        faiss.normalize_L2(src_emb)

        # FAISS search
        dists, indices = self.faiss_index.search(src_emb, k)

        # Temporally-smoothed selection
        selected = np.zeros(T, dtype=np.int64)
        selected[0] = indices[0, 0]
        for t in range(1, T):
            cands = indices[t]
            sem_scores = dists[t]
            temp_scores = 1.0 / (1.0 + np.abs(cands - selected[t-1]).astype(np.float32))
            combined = alpha * sem_scores + (1 - alpha) * temp_scores
            selected[t] = cands[np.argmax(combined)]

        # Build converted token sequence
        out = src_codes.clone()
        for t in range(T):
            out[0, :, t] = torch.tensor(self.full_arr[selected[t]])

        # Decode
        converted = decode(out)
        elapsed = time.time() - t0
        audio_dur = len(source_wav) / SR
        rtf = elapsed / audio_dur

        print(f'Converted {audio_dur:.1f}s audio in {elapsed:.2f}s (RTF={rtf:.4f})')
        return converted


# Create engine
engine = MimiVCEngine()
print('Mimi-VC engine created.')

---
## Step 1: Provide Target Speaker Audio

Record or upload audio of the **target speaker** — the voice you want to clone.

**Tips for best results:**
- Use **clean speech** (minimal background noise)
- **More audio = better quality** (30s minimum, 2+ minutes recommended)
- Speak naturally and continuously

In [ ]:
# ============================================================
# 5. AUDIO RECORDER UTILITY
# ============================================================
# This creates a reusable record_audio() function that works
# for both target and source audio capture.

from IPython.display import display, HTML, Audio as PlayAudio
from google.colab import output
import base64

def record_audio(save_path, max_seconds=120, label="Audio"):
    """
    Record audio directly in the browser via microphone.
    Displays Start/Stop buttons and a timer. When stopped,
    the audio is saved to save_path as a WAV file.

    Args:
        save_path: where to save the recorded WAV
        max_seconds: maximum recording length
        label: display label
    Returns:
        numpy audio array at SR sample rate, or None if failed
    """
    # Unique ID for this recorder instance
    import random
    rec_id = f"rec_{random.randint(10000,99999)}"

    display(HTML(f'''
<div id="{rec_id}_container" style="border:1px solid #ccc; padding:15px; border-radius:8px; max-width:500px; margin:10px 0;">
  <h4 style="margin-top:0;">Record {label}</h4>
  <button id="{rec_id}_start" onclick="{rec_id}_startRec()"
    style="background:#4CAF50; color:white; padding:10px 24px; border:none; border-radius:4px; cursor:pointer; font-size:14px; margin-right:8px;">
    Start Recording
  </button>
  <button id="{rec_id}_stop" onclick="{rec_id}_stopRec()" disabled
    style="background:#f44336; color:white; padding:10px 24px; border:none; border-radius:4px; cursor:pointer; font-size:14px;">
    Stop Recording
  </button>
  <p id="{rec_id}_status" style="margin:10px 0 5px; font-weight:bold;">Click Start to begin.</p>
  <p id="{rec_id}_timer" style="font-family:monospace; font-size:18px; margin:0;">0:00</p>
  <audio id="{rec_id}_playback" controls style="display:none; margin-top:10px; width:100%;"></audio>
</div>
<script>
var {rec_id}_mr, {rec_id}_chunks, {rec_id}_interval, {rec_id}_sec;
async function {rec_id}_startRec() {{
  const stream = await navigator.mediaDevices.getUserMedia({{audio: {{sampleRate: 48000, channelCount: 1, echoCancellation: true, noiseSuppression: true}}}});
  {rec_id}_chunks = [];
  {rec_id}_mr = new MediaRecorder(stream, {{mimeType: 'audio/webm;codecs=opus'}});
  {rec_id}_mr.ondataavailable = e => {rec_id}_chunks.push(e.data);
  {rec_id}_mr.onstop = () => {{
    const blob = new Blob({rec_id}_chunks, {{type: 'audio/webm'}});
    const url = URL.createObjectURL(blob);
    document.getElementById('{rec_id}_playback').src = url;
    document.getElementById('{rec_id}_playback').style.display = 'block';
    document.getElementById('{rec_id}_status').textContent = 'Recording complete! Processing...';
    const reader = new FileReader();
    reader.onload = () => {{
      const b64 = reader.result.split(',')[1];
      google.colab.kernel.invokeFunction('notebook.{rec_id}_save', [b64], {{}});
    }};
    reader.readAsDataURL(blob);
    stream.getTracks().forEach(t => t.stop());
  }};
  {rec_id}_sec = 0;
  {rec_id}_interval = setInterval(() => {{
    {rec_id}_sec++;
    const m = Math.floor({rec_id}_sec / 60);
    const s = String({rec_id}_sec % 60).padStart(2, '0');
    document.getElementById('{rec_id}_timer').textContent = m + ':' + s;
    if ({rec_id}_sec >= {max_seconds}) {rec_id}_stopRec();
  }}, 1000);
  {rec_id}_mr.start(1000);
  document.getElementById('{rec_id}_start').disabled = true;
  document.getElementById('{rec_id}_stop').disabled = false;
  document.getElementById('{rec_id}_status').textContent = 'Recording... speak now!';
  document.getElementById('{rec_id}_status').style.color = '#f44336';
}}
function {rec_id}_stopRec() {{
  clearInterval({rec_id}_interval);
  if ({rec_id}_mr && {rec_id}_mr.state === 'recording') {rec_id}_mr.stop();
  document.getElementById('{rec_id}_start').disabled = false;
  document.getElementById('{rec_id}_stop').disabled = true;
}}
</script>
'''))

    # Register callback to receive audio data
    result = {'done': False, 'path': None}
    webm_path = save_path.replace('.wav', '.webm')

    def on_save(b64_data):
        audio_bytes = base64.b64decode(b64_data)
        with open(webm_path, 'wb') as f:
            f.write(audio_bytes)
        result['done'] = True
        result['path'] = webm_path
        print(f'Received {len(audio_bytes)/1024:.1f} KB audio data.')

    output.register_callback(f'notebook.{rec_id}_save', on_save)

    # Wait for recording
    print(f'Waiting for {label} recording...')
    print('(Click Start, speak, then click Stop)')
    import time
    while not result['done']:
        time.sleep(0.5)

    # Convert webm to numpy array
    wav = load_audio(result['path'])
    dur = len(wav) / SR
    sf.write(save_path, wav, SR)
    print(f'Saved: {save_path} ({dur:.1f}s)')
    return wav


print('Audio recorder ready.')

In [ ]:
# ============================================================
# 6. RECORD TARGET SPEAKER (the voice you want to clone)
# ============================================================
# Record the target speaker directly from your microphone.
# Speak for at least 30 seconds for good quality.
# The longer and more varied the speech, the better the result.

target_wav = record_audio(
    save_path='/tmp/target_speaker.wav',
    max_seconds=180,  # up to 3 minutes
    label='Target Speaker (the voice to clone)'
)

target_wavs = [target_wav]
total_duration = len(target_wav) / SR

print(f'\nTarget speaker audio: {total_duration:.1f}s')
if total_duration < 10:
    print('WARNING: Short recording. Try 30s+ for better results.')

print('\nPlayback:')
display(PlayAudio(target_wav, rate=SR))

In [ ]:
# ============================================================
# 6b. (ALTERNATIVE) Upload target audio files instead
# ============================================================
# Run this cell ONLY if recording didn't work or you prefer
# to upload pre-recorded files. Skip if you already recorded above.

print('Upload target speaker audio file(s):')
uploaded = files.upload()

target_wavs = []
total_duration = 0
for fname, data in uploaded.items():
    tmp_path = f'/tmp/target_{fname}'
    with open(tmp_path, 'wb') as f:
        f.write(data)
    wav = load_audio(tmp_path)
    dur = len(wav) / SR
    total_duration += dur
    target_wavs.append(wav)
    print(f'  Loaded: {fname} ({dur:.1f}s)')

print(f'\nTotal target audio: {total_duration:.1f}s')
print('\nPlayback:')
display(PlayAudio(target_wavs[0][:int(SR * 10)], rate=SR))

---
## Step 2: Build Voice Index

This encodes the target speaker audio with Mimi and builds a FAISS retrieval index.
For long audio, it splits into chunks to avoid memory issues.

In [ ]:
# ============================================================
# 7. BUILD TARGET SPEAKER INDEX
# ============================================================

# Split long audio into chunks (30s each) to avoid OOM
CHUNK_SECONDS = 30
chunk_samples = int(CHUNK_SECONDS * SR)

all_chunks = []
for wav in target_wavs:
    for i in range(0, len(wav), chunk_samples):
        chunk = wav[i:i + chunk_samples]
        if len(chunk) > SR:  # at least 1s
            all_chunks.append(chunk)

print(f'Split into {len(all_chunks)} chunk(s) ({CHUNK_SECONDS}s each)')

# Build index
engine.build_index(all_chunks, target_name='Target Speaker')

print(f'\nReady! The voice index contains {engine.n_frames} frames '
      f'({engine.n_frames / FRAME_RATE:.1f}s of speech).')

---
## Step 3: Record Source Audio & Convert

Record the audio you want to **convert** to sound like the target speaker.
This can be your own voice speaking any sentence.

In [ ]:
# ============================================================
# 8. RECORD SOURCE AUDIO AND CONVERT
# ============================================================

source_wav = record_audio(
    save_path='/tmp/source_audio.wav',
    max_seconds=120,
    label='Source Audio (the speech to convert)'
)

src_dur = len(source_wav) / SR
print(f'\nSource audio: {src_dur:.1f}s')
print('\nOriginal source audio:')
display(PlayAudio(source_wav, rate=SR))

# Convert
CHUNK_SEC = 30
chunk_samples = int(CHUNK_SEC * SR)

if src_dur > CHUNK_SEC:
    print(f'\nLong audio ({src_dur:.1f}s). Converting in chunks...')
    converted_chunks = []
    for i in range(0, len(source_wav), chunk_samples):
        chunk = source_wav[i:i + chunk_samples]
        if len(chunk) > SR:
            conv = engine.convert(chunk)
            converted_chunks.append(conv)
    converted_wav = np.concatenate(converted_chunks)
else:
    converted_wav = engine.convert(source_wav)

# Play converted
print(f'\nConverted audio (now sounds like {engine.target_name}):')
display(PlayAudio(converted_wav, rate=SR))

# Save
out_path = '/content/converted_output.wav'
sf.write(out_path, converted_wav, SR)
print(f'\nSaved to: {out_path}')

---
## Step 4: Compare & Visualize

In [ ]:
# ============================================================
# 8. SIDE-BY-SIDE COMPARISON
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(14, 8), constrained_layout=True)

# Waveforms
t_src = np.arange(len(source_wav)) / SR
t_conv = np.arange(len(converted_wav)) / SR
t_tgt = np.arange(len(target_wavs[0])) / SR

axes[0].plot(t_src, source_wav, color='#4472C4', linewidth=0.3)
axes[0].set_title('Source Audio (original voice)', fontsize=11)
axes[0].set_ylabel('Amplitude')
axes[0].set_xlim(0, max(t_src[-1], 0.1))

axes[1].plot(t_conv, converted_wav, color='#C55A11', linewidth=0.3)
axes[1].set_title(f'Converted Audio (sounds like {engine.target_name})', fontsize=11)
axes[1].set_ylabel('Amplitude')
axes[1].set_xlim(0, max(t_conv[-1], 0.1))

tgt_preview = target_wavs[0][:int(SR * min(30, len(target_wavs[0]) / SR))]
t_tgt_p = np.arange(len(tgt_preview)) / SR
axes[2].plot(t_tgt_p, tgt_preview, color='#548235', linewidth=0.3)
axes[2].set_title(f'{engine.target_name} Reference', fontsize=11)
axes[2].set_ylabel('Amplitude')
axes[2].set_xlabel('Time (s)')
axes[2].set_xlim(0, max(t_tgt_p[-1], 0.1))

for ax in axes:
    ax.grid(True, alpha=0.2)

plt.suptitle('Mimi-VC: Voice Conversion Result', fontsize=13)
plt.savefig('/content/comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 9. TOKEN VISUALIZATION
# ============================================================

# Encode source and converted for visualization
vis_len = min(int(SR * 10), len(source_wav), len(converted_wav))
src_codes_vis = encode(source_wav[:vis_len])
conv_codes_vis = encode(converted_wav[:vis_len])
tgt_codes_vis = encode(target_wavs[0][:vis_len])

L = min(src_codes_vis.shape[2], conv_codes_vis.shape[2], tgt_codes_vis.shape[2], 200)

fig, axes = plt.subplots(3, 1, figsize=(14, 7), constrained_layout=True)

for ax, codes, title in zip(axes, 
    [src_codes_vis, conv_codes_vis, tgt_codes_vis],
    ['Source (original)', 'Converted (FAISS+Smooth)', f'{engine.target_name} (reference)']):
    im = ax.imshow(codes[0, :, :L].numpy(), aspect='auto', cmap='viridis', interpolation='nearest')
    ax.set_ylabel('Codebook')
    ax.set_title(title, fontsize=11)
    ax.set_yticks(range(N_CB))
    ax.set_yticklabels([f'CB{i} ({"sem" if i==0 else "aco"})' for i in range(N_CB)])
    plt.colorbar(im, ax=ax, shrink=0.6)

axes[-1].set_xlabel('Frame (12.5 Hz)')
plt.suptitle('Mimi Token Comparison', fontsize=13)
plt.savefig('/content/tokens.png', dpi=150, bbox_inches='tight')
plt.show()

# Show CB0 similarity stats
src_cb0 = set(src_codes_vis[0, 0, :L].numpy().tolist())
conv_cb0 = set(conv_codes_vis[0, 0, :L].numpy().tolist())
tgt_cb0 = set(tgt_codes_vis[0, 0, :L].numpy().tolist())
print(f'CB0 overlap: source-converted={len(src_cb0 & conv_cb0)}/{len(src_cb0)} tokens, '
      f'converted-target={len(conv_cb0 & tgt_cb0)}/{len(conv_cb0)} tokens')

In [ ]:
# ============================================================
# 10. DOWNLOAD RESULT
# ============================================================

print('Playback summary:')
print('\n1. Source (original):')
display(PlayAudio(source_wav, rate=SR))
print(f'\n2. Converted (-> {engine.target_name}):')
display(PlayAudio(converted_wav, rate=SR))
print(f'\n3. {engine.target_name} reference:')
display(PlayAudio(target_wavs[0][:int(SR*15)], rate=SR))

# Download
print(f'\nDownloading converted audio...')
files.download(out_path)

---
## (Optional) Convert More Audio

The target voice index is still loaded. Record another sample to convert.

In [ ]:
# ============================================================
# 11. CONVERT ADDITIONAL RECORDINGS (reuses same target index)
# ============================================================

print(f'Target voice index: {engine.target_name} ({engine.n_frames / FRAME_RATE:.1f}s)')
print('Record another sample to convert:\n')

import random
extra_id = random.randint(1, 9999)
extra_wav = record_audio(
    save_path=f'/tmp/source_extra_{extra_id}.wav',
    max_seconds=120,
    label='Additional Source Audio'
)

dur = len(extra_wav) / SR
print(f'\nOriginal ({dur:.1f}s):')
display(PlayAudio(extra_wav, rate=SR))

# Convert
if dur > 30:
    chunks = []
    for i in range(0, len(extra_wav), chunk_samples):
        c = extra_wav[i:i + chunk_samples]
        if len(c) > SR:
            chunks.append(engine.convert(c))
    conv = np.concatenate(chunks)
else:
    conv = engine.convert(extra_wav)

print(f'\nConverted (-> {engine.target_name}):')
display(PlayAudio(conv, rate=SR))

extra_out = f'/content/converted_extra_{extra_id}.wav'
sf.write(extra_out, conv, SR)

print(f'\nDownloading...')
files.download(extra_out)